# 04 weathermate: practice

Configuration stays outside executable examples → Messages → Client, request, and response are separate objects → A tool schema describes; a dispatcher executes → An agent loop must make progress and stop → Streaming sends partial progress → Evaluate, give feedback, and bound revision → A callback receives a message and history → Request, status, and parsed body

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [1_foundations/community_contributions/4_weathermate_agent/04_weathermate.ipynb](../../../../../../../1_foundations/community_contributions/4_weathermate_agent/04_weathermate.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. Messages: instructions, earlier turns, and the new question

The model request is a list of messages. A role says who is speaking; content holds the text. The system instruction and prior conversation serve different purposes.

**Predict:** How many messages are sent on the follow-up turn?

In [ ]:
instructions = "Explain Python with small examples."
history = [
    {"role": "user", "content": "What is a list?"},
    {"role": "assistant", "content": "An ordered collection of values."},
]
# List addition preserves the order: instruction, old turns, new question.
messages = [{"role": "system", "content": instructions}] + history + [
    {"role": "user", "content": "How is a dictionary different?"}
]
print([item["role"] for item in messages])
assert len(messages) == 4

**Check:** Four: system, user, assistant, user. With empty history, only two remain.

**Change one thing:** Set history to an empty list. Explain what is lost while the system instruction remains.

## 3. Client, request, and response are separate objects

Constructing a client prepares a connection interface. A request invokes it. The returned response contains choices, a message, and its content. This fixture imitates that nested shape.

**Predict:** What does choices[0] select?

In [ ]:
from types import SimpleNamespace
response = SimpleNamespace(choices=[
    SimpleNamespace(message=SimpleNamespace(content="The answer is 4."))
])
answer = response.choices[0].message.content
print(answer)
assert answer == "The answer is 4."

**Check:** The first returned choice. This is simulated response data, not a model call.

**Change one thing:** Add a second choice with different text. Read choices[1] and explain why choosing an index matters.

## 4. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 5. An agent loop must make progress and stop

A loop repeats model/tool steps until a final answer or a configured limit. This fixed script of decisions shows the order without executing generated code or calling a model.

**Predict:** Which decision ends the loop?

In [ ]:
decisions = iter([
    {"kind": "tool", "name": "add", "a": 12, "b": 30},
    {"kind": "answer", "text": "The sum is 42."},
])
events = []
for turn in range(3):  # Bound the number of steps.
    decision = next(decisions)
    if decision["kind"] == "tool":
        if decision["name"] != "add":
            raise ValueError("Unknown tool")
        events.append({"tool_result": decision["a"] + decision["b"]})
    else:
        events.append({"final": decision["text"]})
        break
else:
    raise RuntimeError("No final answer within the limit")
print(events)
assert events[0]["tool_result"] == 42

**Check:** The answer decision ends it. A loop that only keeps requesting tools needs a bound and an explicit failure path.

**Change one thing:** Add another tool step before the answer. Keep a finite turn limit and explain how the stopping rule changes.

## 6. Streaming sends partial progress

A generator yields multiple updates. A streamed API can include multiple event types, so a UI should select the text events rather than assume every event is text.

**Predict:** How many text chunks are selected?

In [ ]:
events = [
    {"type": "status", "value": "started"},
    {"type": "text", "value": "Hello "},
    {"type": "text", "value": "learner"},
]
def text_chunks(items):
    for event in items:
        if event["type"] == "text":
            yield event["value"]
chunks = list(text_chunks(events))
print(chunks, "→", "".join(chunks))
assert "".join(chunks) == "Hello learner"

**Check:** Two. yield preserves progress; return ends an ordinary function.

**Change one thing:** Insert a non-text event between the chunks. Confirm it does not become displayed answer text.

## 7. Evaluate, give feedback, and bound revision

A worker produces a draft and an evaluator judges it against criteria. Feedback becomes input for another attempt. The evaluator here is a small deterministic teaching rule, not a general quality checker.

**Predict:** Is the first draft acceptable? How many attempts will be allowed?

In [ ]:
def evaluate(text):
    return {"accepted": "example:" in text.lower(), "feedback": "Include an example."}
attempts = []
for attempt in range(2):
    draft = "A list stores ordered values." if attempt == 0 else "A list stores ordered values. Example: [1, 2]"
    decision = evaluate(draft)
    attempts.append(decision["accepted"])
    if decision["accepted"]:
        break
print(attempts, draft)
assert attempts == [False, True]

**Check:** The first is rejected and the second accepted. Unlike the Lab 3 one-retry source, this teaching example evaluates the revision too.

**Change one thing:** Make the second draft fail too. Add an explicit message when the finite retry limit is reached.

## 8. A callback receives a message and history

A UI invokes your function and displays its return value. Earlier turns are an input, not automatic model memory. This practice calls the callback directly without starting a server.

**Predict:** What changes when history contains two old messages?

In [ ]:
def chat(message, history):
    return f"You asked: {message}. Earlier messages: {len(history)}."
first = chat("Hello", [])
second = chat("Explain more", [{"role":"user", "content":"Hello"}, {"role":"assistant", "content":first}])
print(first, second, sep="\n")
assert "Earlier messages: 2" in second

**Check:** The count becomes two. Gradio wiring is gr.ChatInterface(chat, type="messages"); launch starts the server.

**Change one thing:** Make the callback return an uppercase version of the question. Compare returning a value with printing it.

## 9. Request, status, and parsed body

An HTTP client sends a request and returns a status plus response body. Provider requests are network side effects; a fixture lets you practice handling success/failure without sending one.

**Predict:** Will a non-200 response be treated as success here?

In [ ]:
fixtures = [{"status": 200, "body": {"result": "fictional success"}},
            {"status": 401, "body": {"error": "authentication rejected"}}]
for response in fixtures:
    if response["status"] >= 400:
        print("Failure status:", response["status"])
    else:
        print("Parsed result:", response["body"]["result"])
assert fixtures[1]["status"] == 401

**Check:** Status values below 400 enter this fixture success branch; 401 is rejected. Use the provider response to distinguish causes.

**Change one thing:** Add a 429 fixture and distinguish rate limiting from authentication. Explain why real calls need timeouts.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.